# ATLAS Group Notebook: Finding the Higgs Boson
## H → ZZ* → 4ℓ: The Four-Lepton "Golden Channel"

**Vanderbilt Programs for Talented Youth | Instructor: Jennifer James, Ph.D. Candidate**

---

### The challenge

The Higgs boson was discovered in 2012 by the ATLAS and CMS experiments at the LHC. Alongside H → γγ, the other main discovery channel was **H → ZZ\* → 4ℓ**: the Higgs decaying to two Z bosons (one on-shell, one off-shell, written Z\*), each of which decays to a charged lepton pair (electrons or muons).

This channel is even rarer than diphoton — it happens only about 0.01% of the time. But it is **exceptionally clean**: four charged leptons whose combined invariant mass reconstructs to a sharp peak at 125 GeV, sitting on a very small, well-understood background.

**Why this channel matters for our question:** photons are massless and don't couple to the Higgs field directly — H → γγ proceeds through a virtual loop of W bosons or top quarks. Leptons are different. Electrons and muons get their mass *directly* from the Higgs field, through a Yukawa coupling:

$$m_\ell = \frac{y_\ell \, v}{\sqrt{2}}$$

where $v \approx 246$ GeV is the Higgs vacuum expectation value (the "on" value of the field) and $y_\ell$ is the lepton's Yukawa coupling strength. So when we reconstruct a Higgs peak using leptons, we are looking at particles whose very mass is evidence that the Higgs field has a nonzero value everywhere in space.

Your detector is redesigned to find exactly this signal. This notebook walks through the same statistical challenge as before: extracting a small signal from a background. The technique is identical to what the real ATLAS analysis did.

---

$$m_{4\ell}^2 = \left(\sum_{i=1}^{4} E_i\right)^2 - \left|\sum_{i=1}^{4} \vec{p}_i\right|^2$$

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.stats import chi2

np.random.seed(2012)  # The year of the discovery
print("Libraries loaded!")
print("Seed set to 2012, the very year the Higgs was discovered.")

---

## Part 1: Generating the Four-Lepton Mass Spectrum

We simulate the four-lepton invariant mass spectrum ($m_{4\ell}$) as ATLAS would see it:

- **Background**: a small, gently falling continuum, mostly from ZZ\* production that doesn't involve a Higgs at all (q+q̄ → ZZ\*), plus a tiny contribution from other processes
- **Signal**: a narrow Gaussian peak at 125 GeV from H → ZZ\* → 4ℓ. The width comes from the **tracker's momentum resolution**, since lepton energies here are reconstructed from track curvature, not calorimeter showers

Notice how much smaller the background is compared to the diphoton channel — requiring four leptons in coincidence is a very effective filter against non-Higgs events.

In [ ]:
# ── Mass range and binning ─────────────────────────────────────────────────────────────
m_bins    = np.linspace(100, 160, 61)   # 1 GeV bins, 100-160 GeV
m_centers = 0.5 * (m_bins[:-1] + m_bins[1:])
m_higgs   = 125.09  # GeV/c2 (PDG 2023 value)

# ── Background model: gently falling ZZ* continuum ───────────────────────────────
def background(m, N_bkg, a, b):
    """Continuum background model (ZZ* production, no Higgs involved)."""
    return N_bkg * np.exp(a * (m - 100) + b * (m - 100)**2)

N_bkg  = 300
a_true = -0.02
b_true = -0.0001

bkg_rate = background(m_centers, N_bkg, a_true, b_true)
bkg_rate = bkg_rate / bkg_rate.sum() * N_bkg

# ── Signal model: Gaussian at 125 GeV ───────────────────────────────────────
N_signal  = 30     # Higgs events (realistic scale for early LHC run, 4-lepton channel)
sigma_det = 2.2    # GeV (tracker momentum resolution, propagated to m_4l)

sig_rate = N_signal * np.exp(-0.5 * ((m_centers - m_higgs)/sigma_det)**2)
sig_rate /= sig_rate.sum()
sig_rate *= N_signal

# ── Generate observed data (Poisson fluctuations) ─────────────────────────────
total_rate   = bkg_rate + sig_rate
observed     = np.random.poisson(total_rate)
observed_err = np.sqrt(np.maximum(observed, 1))

print(f"Simulated four-lepton mass spectrum:")
print(f"  Total events:    {observed.sum():,}")
print(f"  Background:      ~{int(N_bkg):,}")
print(f"  Signal (Higgs):  ~{N_signal} events at {m_higgs:.2f} GeV")
print(f"  Signal/Total:    {N_signal/(N_bkg+N_signal)*100:.1f}%")
print(f"  Tracker-driven mass resolution: σ = {sigma_det} GeV")

---

## Part 2: The Raw Spectrum: Can You See the Higgs?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.errorbar(m_centers, observed, yerr=observed_err,
            fmt='o', color='black', markersize=5, capsize=3,
            linewidth=1, label='Data (observed)')
ax.plot(m_centers, bkg_rate + sig_rate, color='firebrick',
        linewidth=2, label='True signal + background (hidden in real life!)', alpha=0.6)
ax.set_xlabel(r'Four-lepton invariant mass $M_{4\ell}$ (GeV/c$^2$)', fontsize=12)
ax.set_ylabel('Events / 1 GeV', fontsize=12)
ax.set_title('Raw four-lepton mass spectrum\nCan you see the Higgs by eye?', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(100, 160)
ax.axvline(x=m_higgs, color='gold', linestyle='--', linewidth=1.5, alpha=0.6)

ax = axes[1]
zoom_mask = (m_centers >= 115) & (m_centers <= 135)
ax.errorbar(m_centers[zoom_mask], observed[zoom_mask], yerr=observed_err[zoom_mask],
            fmt='o', color='black', markersize=6, capsize=3, linewidth=1.5, label='Data')
ax.plot(m_centers[zoom_mask], (bkg_rate+sig_rate)[zoom_mask],
        color='firebrick', linewidth=2, label='True total (hidden)', alpha=0.6)
ax.set_xlabel(r'$M_{4\ell}$ (GeV/c$^2$)', fontsize=12)
ax.set_ylabel('Events / 1 GeV', fontsize=12)
ax.set_title('Zoomed: 115-135 GeV\nThe Higgs signal region', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.suptitle('Simulated ATLAS four-lepton spectrum (H -> ZZ* -> 4l, inspired by 2012 discovery)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print("Unlike the diphoton channel, the background here is already small.")
print("But with so few total events, statistical fluctuations still make the peak hard to confirm by eye.")
print("We still need a background subtraction method to be rigorous about it.")

---

## Part 3: Background Subtraction: Finding the Bump

The real ATLAS analysis fits a smooth function to the background **excluding** the signal region, then subtracts it. What's left is the signal.

We do the same thing here.

In [ ]:
sideband_mask = (m_centers < 120) | (m_centers > 130)
m_fit   = m_centers[sideband_mask]
obs_fit = observed[sideband_mask]
err_fit = observed_err[sideband_mask]

def bkg_model(m, N, a, b):
    return N * np.exp(a*(m-100) + b*(m-100)**2)

popt, pcov = curve_fit(bkg_model, m_fit, obs_fit, p0=[300, -0.02, -0.0001],
                        sigma=err_fit, absolute_sigma=True)
perr = np.sqrt(np.diag(pcov))

bkg_fitted = bkg_model(m_centers, *popt)
residuals  = observed - bkg_fitted

fig, axes = plt.subplots(2, 1, figsize=(10, 8),
                          gridspec_kw={'height_ratios': [3, 1]})

ax = axes[0]
ax.errorbar(m_centers, observed, yerr=observed_err,
            fmt='o', color='black', markersize=5, capsize=3, label='Data', zorder=4)
ax.plot(m_centers, bkg_fitted, color='steelblue', linewidth=2.5,
        label='Background fit (sideband method)', zorder=3)
ax.fill_between(m_centers, bkg_fitted - perr[0], bkg_fitted + perr[0],
                alpha=0.2, color='steelblue', label='Fit uncertainty')
ax.axvspan(120, 130, alpha=0.08, color='gold', label='Signal region (excluded from fit)')
ax.axvline(x=m_higgs, color='gold', linestyle='--', linewidth=1.5)
ax.set_ylabel('Events / 1 GeV', fontsize=12)
ax.set_title('Background subtraction: sideband fit method\n'
             'Fit the background outside the signal region, then subtract', fontsize=11)
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim(100, 160)

ax = axes[1]
ax.errorbar(m_centers, residuals, yerr=observed_err,
            fmt='o', color='firebrick', markersize=5, capsize=3, label='Data - background')
ax.axhline(y=0, color='black', linewidth=1.5)
ax.axvline(x=m_higgs, color='gold', linestyle='--', linewidth=1.5)
ax.axvspan(120, 130, alpha=0.08, color='gold')
ax.set_xlabel(r'$M_{4\ell}$ (GeV/c$^2$)', fontsize=12)
ax.set_ylabel('Residuals', fontsize=12)
ax.set_title('Residuals = Data - Background fit  ->  The Higgs signal!', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(100, 160)

plt.tight_layout()
plt.show()

---

## Part 4: Measuring the Higgs: Signal Extraction

In [ ]:
signal_mask = (m_centers >= 115) & (m_centers <= 135)
m_sig   = m_centers[signal_mask]
res_sig = residuals[signal_mask]
err_sig = observed_err[signal_mask]

def gaussian(m, N, mu, sigma):
    return N * np.exp(-0.5*((m - mu)/sigma)**2)

try:
    popt_sig, pcov_sig = curve_fit(gaussian, m_sig, res_sig,
                                    p0=[20, 125, 2.0], sigma=err_sig,
                                    absolute_sigma=True)
    perr_sig = np.sqrt(np.diag(pcov_sig))
    fit_success = True
except Exception as e:
    print(f"Fit failed: {e}")
    fit_success = False

fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(m_sig, res_sig, yerr=err_sig,
            fmt='o', color='black', markersize=7, capsize=4, label='Residuals (data - bkg)')
if fit_success:
    m_fine = np.linspace(115, 135, 200)
    ax.plot(m_fine, gaussian(m_fine, *popt_sig), color='firebrick',
            linewidth=2.5, label='Gaussian fit')
    ax.fill_between(m_fine, 0, gaussian(m_fine, *popt_sig), alpha=0.2, color='firebrick')
ax.axhline(y=0, color='gray', linewidth=1)
ax.axvline(x=m_higgs, color='gold', linestyle='--', linewidth=2,
           label=f'PDG Higgs mass: {m_higgs} GeV')
ax.set_xlabel(r'$M_{4\ell}$ (GeV/c$^2$)', fontsize=13)
ax.set_ylabel('Events / 1 GeV (background subtracted)', fontsize=12)
ax.set_title('The Higgs boson signal\nBackground-subtracted four-lepton mass spectrum', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

if fit_success:
    N_fit, mu_fit, sig_fit = popt_sig
    N_err, mu_err, sig_err = perr_sig
    significance = N_fit / N_err
    print(f"Fitted Higgs mass:        {mu_fit:.2f} +/- {mu_err:.2f} GeV/c2")
    print(f"Fitted mass resolution:   {abs(sig_fit):.2f} +/- {sig_err:.2f} GeV  (tracker resolution)")
    print(f"Fitted signal yield:      {N_fit:.0f} +/- {N_err:.0f} events")
    print(f"Statistical significance: {significance:.1f} sigma")
    print()
    print(f"True Higgs mass (PDG): {m_higgs} GeV/c2")
    print(f"Difference: {abs(mu_fit - m_higgs):.2f} GeV  ({abs(mu_fit-m_higgs)/m_higgs*100:.2f}%)")
    print()
    print("The real 2012 ATLAS 4-lepton analysis, combined with diphoton, contributed to the")
    print("overall 5.9 sigma discovery significance. 5 sigma is the conventional threshold")
    print("for a 'discovery' in particle physics.")

---

## Part 5: Why Tracker Momentum Resolution Matters

In the diphoton channel, calorimeter *energy* resolution sets the width of the peak. Here, the four lepton momenta are measured from the **curvature of their tracks** in the detector's magnetic field:

$$p = qBr$$

A higher-momentum lepton curves less, so a bigger tracker radius (a longer lever arm) or a stronger magnetic field is needed to measure that curvature — and therefore the momentum — precisely. Your detector design choices (tracker size, magnetic field strength, number/precision of hit layers) directly control how sharp the Higgs peak looks in $M_{4\ell}$.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

resolutions = [
    (1.2, 'Excellent (sigma = 1.2 GeV) (large radius, strong B field)'),
    (2.2, 'Good (sigma = 2.2 GeV) (ATLAS-like inner detector)'),
    (4.0, 'Poor (sigma = 4.0 GeV) (small radius or weak B field)'),
    (7.0, 'Very poor (sigma = 7.0 GeV) (minimal tracking, few hit layers)'),
]
colors_res = ['steelblue', 'firebrick', 'darkorange', 'gray']

m_fine = np.linspace(115, 135, 300)
for (sigma, label), color in zip(resolutions, colors_res):
    peak = N_signal * np.exp(-0.5*((m_fine - m_higgs)/sigma)**2)
    peak /= peak.max()
    ax.plot(m_fine, peak, linewidth=2.5, color=color,
            linestyle='-' if sigma <= 2.2 else '--', label=label)

ax.axvline(x=m_higgs, color='gold', linestyle=':', linewidth=1.5)
ax.set_xlabel(r'$M_{4\ell}$ (GeV/c$^2$)', fontsize=13)
ax.set_ylabel('Signal peak (normalized to 1)', fontsize=12)
ax.set_title('Effect of tracker momentum resolution on the Higgs peak\n'
             'Worse resolution = broader, shorter peak = harder to find', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("With poor momentum resolution, the Higgs peak smears out across many bins.")
print("The signal-to-background ratio in any single bin drops.")
print("At sigma = 7 GeV, the Higgs peak would be essentially invisible.")
print()
print("This is why your redesigned detector must prioritize tracker radius and magnetic field strength.")

---

## Your Group's Checkpoint Questions

1. In Part 1, the leptons in this channel get their mass directly from the Higgs field through a Yukawa coupling, unlike the photons in the diphoton channel. Why does that make H -> ZZ* -> 4l a more direct probe of "the Higgs field turning on and giving particles mass" than H -> gamma gamma?

2. Look at the sideband fit in Part 3. Why is it important to **exclude** the signal region (120-130 GeV) when fitting the background? What would go wrong if you included it?

3. Compare the background scale in Part 1 (N_bkg = 300) to the diphoton notebook's background (N_bkg = 8000). Why is the four-lepton background so much smaller, even though the signal is also smaller?

4. In Part 5, compare the sigma = 1.2 GeV and sigma = 7.0 GeV resolution curves. Roughly how much taller is the sharp peak compared to the broad one at the peak center? Why does this matter for signal-to-background?

5. From your subsystem checklist, you decided which ATLAS subsystems to keep and which to drop for H -> ZZ* -> 4l. Does anything in this notebook change your answer? Specifically, is the magnet *more* or *less* important for this channel than it was for diphoton, and why?

6. The real 2012 discovery combined significance from *both* the diphoton and four-lepton channels rather than relying on either alone. Why might combining two independent, very different channels give physicists more confidence than either channel by itself?